# 🤖 Crypto Agent — Backtest Expanding Window

Simula exactamente lo que haría el sistema en producción, **sin mirar el futuro**:

- **HMM re-entrenado cada 90 días** con datos disponibles hasta ese momento
- **Fear & Greed histórico** como filtro de sentiment
- **Salida por cambio de régimen** (LONG + BEAR_TREND → cierre de mercado)
- **Max 1 posición por par** simultánea
- Sin llamadas a Claude (replicamos la lógica del system prompt)

Datos desde Google Drive (ya descargados previamente).

In [ ]:
# ── 0. Instalar dependencias ──────────────────────────────────
!pip install hmmlearn scikit-learn -q

In [ ]:
# ── 1. Descargar datos desde Binance (alternativa sin Drive) ─
# Descarga ~3 años de velas 4h directamente desde la API pública de Binance
# No necesita cuenta ni API key. Guarda CSV en /content/

import os
import time
import requests
import pandas as pd

SYMBOLS  = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'XRPUSDT']
INTERVAL = '4h'
DATA_PATH = '/content/'   # se usa como DRIVE_PATH en el resto del notebook
DRIVE_PATH = DATA_PATH    # alias para compatibilidad


def download_binance_ohlcv(symbol: str, interval: str = '4h', years: int = 3) -> pd.DataFrame:
    """Descarga OHLCV de Binance en lotes de 1000 barras."""
    base_url = 'https://api.binance.com/api/v3/klines'
    ms_per_bar = {'1h': 3_600_000, '4h': 14_400_000, '1d': 86_400_000}[interval]
    start_ms = int((pd.Timestamp.now() - pd.Timedelta(days=years*365)).timestamp() * 1000)
    end_ms   = int(pd.Timestamp.now().timestamp() * 1000)

    all_bars = []
    cursor   = start_ms

    while cursor < end_ms:
        r = requests.get(base_url, params={
            'symbol':    symbol,
            'interval':  interval,
            'startTime': cursor,
            'limit':     1000,
        }, timeout=15)
        bars = r.json()
        if not bars or isinstance(bars, dict):  # error de API
            print(f'  ERROR: {bars}')
            break
        all_bars.extend(bars)
        cursor = bars[-1][0] + ms_per_bar  # siguiente barra
        time.sleep(0.2)  # respetar rate limit

    df = pd.DataFrame(all_bars, columns=[
        'open_time','open','high','low','close','volume',
        'close_time','quote_vol','trades','taker_base','taker_quote','ignore'
    ])
    df['open_time'] = pd.to_datetime(df['open_time'], unit='ms', utc=True).dt.tz_localize(None)
    df = df.set_index('open_time')[['open','high','low','close','volume']].astype(float)
    df = df[~df.index.duplicated()].sort_index()
    return df


# ── Descargar y guardar ──
for sym in SYMBOLS:
    fpath = os.path.join(DATA_PATH, f'{sym}_{INTERVAL}.csv')
    if os.path.exists(fpath):
        print(f'  {sym}: ya existe, saltando')
        continue
    print(f'  Descargando {sym} {INTERVAL}...', end=' ')
    df = download_binance_ohlcv(sym, INTERVAL, years=3)
    df.to_csv(fpath)
    print(f'{len(df)} barras | {df.index[0].date()} → {df.index[-1].date()}')

print('
Datos listos en', DATA_PATH)


In [ ]:
# ── 2. Imports ────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from dataclasses import dataclass
from sklearn.preprocessing import StandardScaler
from hmmlearn.hmm import GaussianHMM
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
plt.rcParams.update({'figure.facecolor': '#0f1424', 'axes.facecolor': '#161b2e',
                     'grid.color': '#1e2640', 'text.color': '#e2e8f0'})

In [ ]:
# ── 3. Fear & Greed histórico ─────────────────────────────────
import requests

def fetch_fear_greed_history(days: int = 730) -> pd.Series:
    """Descarga historial de Fear & Greed Index. Retorna Serie indexada por fecha."""
    r    = requests.get(f'https://api.alternative.me/fng/?limit={days}', timeout=15)
    data = r.json()['data']
    df   = pd.DataFrame(data)
    df['timestamp'] = pd.to_datetime(df['timestamp'].astype(int), unit='s').dt.normalize()
    df['value']     = df['value'].astype(int)
    df = df.set_index('timestamp').sort_index()
    return df['value']

fng_history = fetch_fear_greed_history(730)
print(f'Fear & Greed: {len(fng_history)} días | {fng_history.index[0].date()} → {fng_history.index[-1].date()}')
print(f'Media: {fng_history.mean():.1f} | Min: {fng_history.min()} | Max: {fng_history.max()}')

In [ ]:
# ── 4. Features HMM (idéntico al agente en producción) ────────
def compute_hmm_features(df: pd.DataFrame):
    """5 features: log_return, vol_20, vol_ratio, rsi_centered, ema50_slope"""
    d = df.copy()
    d['log_ret']     = np.log(d['close'] / d['close'].shift(1))
    d['vol_20']      = d['log_ret'].rolling(20).std()
    vol_ma           = d['volume'].rolling(20).mean().replace(0, 1e-10)
    d['vol_ratio']   = d['volume'] / vol_ma
    delta            = d['close'].diff()
    gain             = delta.clip(lower=0).rolling(14).mean()
    loss             = (-delta.clip(upper=0)).rolling(14).mean().replace(0, 1e-10)
    d['rsi_c']       = (100 - 100 / (1 + gain / loss) - 50) / 50
    d['ema50']       = d['close'].ewm(span=50).mean()
    d['ema50_slope'] = d['ema50'].diff(3) / d['ema50']
    d.dropna(inplace=True)
    cols = ['log_ret', 'vol_20', 'vol_ratio', 'rsi_c', 'ema50_slope']
    return d[cols].values, d.index


def label_states(model, scaler, n_components):
    """Etiqueta estados por media de log_return y volatilidad."""
    means      = scaler.inverse_transform(model.means_)
    states     = list(range(n_components))
    by_return  = sorted(states, key=lambda i: means[i][0])
    bear, bull = by_return[0], by_return[-1]
    middle     = by_return[1:-1]
    labels     = {bull: 'BULL_TREND', bear: 'BEAR_TREND'}
    if len(middle) >= 2:
        sideways, reversal = sorted(middle, key=lambda i: means[i][1])
        labels[sideways]   = 'SIDEWAYS'
        labels[reversal]   = 'REVERSAL'
    elif len(middle) == 1:
        labels[middle[0]]  = 'SIDEWAYS'
    return labels


def train_hmm(X_scaled, n_components=4, n_restarts=5):
    best, best_score = None, -np.inf
    for seed in range(n_restarts):
        m = GaussianHMM(n_components=n_components, covariance_type='full',
                        n_iter=200, random_state=seed, tol=1e-4)
        try:
            m.fit(X_scaled)
            s = m.score(X_scaled)
            if s > best_score:
                best, best_score = m, s
        except Exception:
            continue
    return best

print('Funciones HMM definidas ✓')

In [ ]:
# ── 5. Indicadores técnicos ───────────────────────────────────
@dataclass
class StrategyParams:
    rsi_period:      int   = 14
    ema_fast:        int   = 20
    ema_slow:        int   = 50
    stop_loss_pct:   float = 0.04
    take_profit_pct: float = 0.08


def calc_indicators(df: pd.DataFrame, p: StrategyParams) -> pd.DataFrame:
    df = df.copy()
    delta        = df['close'].diff()
    gain         = delta.clip(lower=0).rolling(p.rsi_period).mean()
    loss         = (-delta.clip(upper=0)).rolling(p.rsi_period).mean().replace(0, 1e-10)
    df['rsi']    = 100 - (100 / (1 + gain / loss))
    df['ema_f']  = df['close'].ewm(span=p.ema_fast).mean()
    df['ema_s']  = df['close'].ewm(span=p.ema_slow).mean()
    df['trend']  = df['ema_f'] > df['ema_s']
    return df


def get_fng_for_bar(ts: pd.Timestamp, fng_series: pd.Series) -> int:
    """Busca el F&G del día de la vela. Si no existe, forward-fill."""
    day = ts.normalize()
    if day in fng_series.index:
        return int(fng_series[day])
    # buscar el último disponible antes de esa fecha
    past = fng_series[fng_series.index <= day]
    return int(past.iloc[-1]) if len(past) else 50

print('Indicadores definidos ✓')

In [ ]:
# ── 6. Generador de señales con régimen + F&G ─────────────────
# Replica el system prompt de Claude sin llamar a la API

REGIME_BIAS = {
    'BULL_TREND': {'allow_long': True,  'allow_short': False, 'rsi_ob': 70, 'rsi_os': 40},
    'BEAR_TREND': {'allow_long': False, 'allow_short': True,  'rsi_ob': 60, 'rsi_os': 30},
    'SIDEWAYS':   {'allow_long': False, 'allow_short': False, 'rsi_ob': 65, 'rsi_os': 35},
    'REVERSAL':   {'allow_long': True,  'allow_short': False, 'rsi_ob': 60, 'rsi_os': 40},
}

def generate_signal(row, regime: str, fng: int) -> str:
    """Señal de entrada con filtros: régimen + RSI + Fear & Greed."""
    if pd.isna(row['rsi']) or pd.isna(row['ema_f']):
        return 'NEUTRAL'

    # Filtro Fear & Greed (reglas del system prompt)
    if fng < 20:   # Extreme Fear → solo LONGs
        if row['trend'] == False:
            return 'NEUTRAL'
    if fng > 80:   # Extreme Greed → sesgo bajista, no entrar LONG
        if row['trend'] == True:
            return 'NEUTRAL'

    bias = REGIME_BIAS.get(regime, {'allow_long': False, 'allow_short': False,
                                     'rsi_ob': 70, 'rsi_os': 30})
    if bias['allow_long']  and row['trend'] and row['rsi'] < bias['rsi_ob']:
        return 'LONG'
    if bias['allow_short'] and not row['trend'] and row['rsi'] > bias['rsi_os']:
        return 'SHORT'
    return 'NEUTRAL'

print('Generador de señales definido ✓')

In [ ]:
# ── 7. Motor de simulación expanding window ───────────────────

def run_expanding_window(
    df_4h:       pd.DataFrame,
    symbol:      str,
    fng_series:  pd.Series,
    p:           StrategyParams,
    warmup_days: int = 180,   # datos mínimos para entrenar HMM (6 meses)
    retrain_days:int = 90,    # re-entrenar HMM cada 90 días
    n_components:int = 4,
) -> tuple[list, list]:
    """
    Simula trades en expanding window sin look-ahead.
    Retorna (trades, regime_log).
    """
    df   = calc_indicators(df_4h, p)
    df.dropna(inplace=True)

    trades      = []
    regime_log  = []   # (timestamp, regime)
    in_trade    = False
    trade       = None
    model       = None
    scaler      = None
    labels      = {}
    last_trained_bar = 0
    bars_per_day     = 6   # 4h → 6 velas por día
    warmup_bars      = warmup_days * bars_per_day
    retrain_bars     = retrain_days * bars_per_day

    for i in range(warmup_bars, len(df)):
        row = df.iloc[i]
        ts  = df.index[i]

        # ── Re-entrenar HMM si corresponde ──
        if model is None or (i - last_trained_bar) >= retrain_bars:
            train_df = df_4h.iloc[:i]   # solo datos pasados
            X, _     = compute_hmm_features(train_df)
            if len(X) > 100:
                sc        = StandardScaler()
                X_sc      = sc.fit_transform(X)
                m         = train_hmm(X_sc, n_components)
                if m is not None:
                    model, scaler, labels = m, sc, label_states(m, sc, n_components)
                    last_trained_bar = i

        # ── Predecir régimen actual ──
        regime = 'UNKNOWN'
        if model is not None:
            X_cur, _ = compute_hmm_features(df_4h.iloc[max(0, i-100):i+1])
            if len(X_cur) > 0:
                X_sc     = scaler.transform(X_cur)
                state    = int(model.predict(X_sc)[-1])
                regime   = labels.get(state, 'UNKNOWN')
        regime_log.append({'ts': ts, 'regime': regime})

        # ── Monitor posición abierta ──
        if in_trade and trade:
            high, low = row['high'], row['low']

            # Salida por cambio de régimen
            regime_exit = (
                (trade['direction'] == 'LONG'  and regime == 'BEAR_TREND') or
                (trade['direction'] == 'SHORT' and regime == 'BULL_TREND')
            )

            if regime_exit:
                exit_price     = row['close']
                trade['exit_price']  = exit_price
                trade['exit_time']   = ts
                trade['exit_reason'] = f'regime_change→{regime}'
                if trade['direction'] == 'LONG':
                    trade['pnl_pct'] = (exit_price - trade['entry_price']) / trade['entry_price'] * 100
                else:
                    trade['pnl_pct'] = (trade['entry_price'] - exit_price) / trade['entry_price'] * 100
                trade['result'] = 'WIN' if trade['pnl_pct'] > 0 else 'LOSS'
                trades.append(trade)
                in_trade = False

            elif trade['direction'] == 'LONG':
                if low <= trade['stop_loss']:
                    trade.update({'exit_price': trade['stop_loss'], 'exit_time': ts,
                                  'result': 'LOSS', 'exit_reason': 'stop',
                                  'pnl_pct': (trade['stop_loss'] - trade['entry_price']) / trade['entry_price'] * 100})
                    trades.append(trade); in_trade = False
                elif high >= trade['take_profit']:
                    trade.update({'exit_price': trade['take_profit'], 'exit_time': ts,
                                  'result': 'WIN', 'exit_reason': 'target',
                                  'pnl_pct': (trade['take_profit'] - trade['entry_price']) / trade['entry_price'] * 100})
                    trades.append(trade); in_trade = False
            else:  # SHORT
                if high >= trade['stop_loss']:
                    trade.update({'exit_price': trade['stop_loss'], 'exit_time': ts,
                                  'result': 'LOSS', 'exit_reason': 'stop',
                                  'pnl_pct': (trade['entry_price'] - trade['stop_loss']) / trade['entry_price'] * 100})
                    trades.append(trade); in_trade = False
                elif low <= trade['take_profit']:
                    trade.update({'exit_price': trade['take_profit'], 'exit_time': ts,
                                  'result': 'WIN', 'exit_reason': 'target',
                                  'pnl_pct': (trade['entry_price'] - trade['take_profit']) / trade['entry_price'] * 100})
                    trades.append(trade); in_trade = False

        # ── Buscar entrada si libre ──
        if not in_trade:
            fng    = get_fng_for_bar(ts, fng_series)
            signal = generate_signal(row, regime, fng)
            if signal != 'NEUTRAL':
                price  = row['close']
                stop   = price * (1 - p.stop_loss_pct)   if signal == 'LONG' else price * (1 + p.stop_loss_pct)
                target = price * (1 + p.take_profit_pct) if signal == 'LONG' else price * (1 - p.take_profit_pct)
                trade  = {'symbol': symbol, 'direction': signal, 'entry_price': price,
                          'stop_loss': stop, 'take_profit': target, 'entry_time': ts,
                          'regime_at_entry': regime, 'fng_at_entry': fng}
                in_trade = True

    return trades, regime_log

print('Motor expanding window definido ✓')

In [ ]:
# ── 8. Métricas ───────────────────────────────────────────────
def calc_metrics(trades: list, symbol: str) -> dict:
    if not trades:
        return {'symbol': symbol, 'trades': 0}
    wins   = [t for t in trades if t['result'] == 'WIN']
    losses = [t for t in trades if t['result'] == 'LOSS']
    total  = len(trades)

    returns   = pd.Series([t['pnl_pct'] for t in trades])
    sharpe    = (returns.mean() / returns.std() * np.sqrt(252)) if returns.std() > 0 else 0
    win_rate  = len(wins) / total * 100
    expectancy= (win_rate/100 * np.mean([t['pnl_pct'] for t in wins]  or [0])) + \
                ((1-win_rate/100) * np.mean([t['pnl_pct'] for t in losses] or [0]))

    # Equity curve (2% riesgo por trade)
    capital, peak, dds = 1000.0, 1000.0, []
    equity = [capital]
    for t in trades:
        capital += capital * (t['pnl_pct'] / 100) * 0.02
        equity.append(capital)
        peak = max(peak, capital)
        dds.append((peak - capital) / peak * 100)

    # Trades por motivo de salida
    exit_reasons = {}
    for t in trades:
        r = t.get('exit_reason', 'unknown')
        exit_reasons[r] = exit_reasons.get(r, 0) + 1

    # Trades por régimen de entrada
    regime_counts = {}
    for t in trades:
        r = t.get('regime_at_entry', 'UNKNOWN')
        regime_counts[r] = regime_counts.get(r, 0) + 1

    return {
        'symbol':        symbol,
        'trades':        total,
        'wins':          len(wins),
        'losses':        len(losses),
        'win_rate':      round(win_rate, 1),
        'avg_win':       round(np.mean([t['pnl_pct'] for t in wins])  if wins   else 0, 2),
        'avg_loss':      round(np.mean([t['pnl_pct'] for t in losses]) if losses else 0, 2),
        'expectancy':    round(expectancy, 3),
        'sharpe':        round(sharpe, 2),
        'max_drawdown':  round(max(dds) if dds else 0, 1),
        'return_pct':    round((capital - 1000) / 1000 * 100, 1),
        'final_capital': round(capital, 2),
        'equity':        equity,
        'exit_reasons':  exit_reasons,
        'regime_counts': regime_counts,
    }

print('Métricas definidas ✓')

In [ ]:
# ── 9. CORRER EL BACKTEST ─────────────────────────────────────
# ⚠️  Tarda ~5-10 min por par (re-entrena HMM varias veces)

p = StrategyParams()  # parámetros del agente actual

all_results = {}

for symbol in SYMBOLS:
    fname = f'{symbol}_{INTERVAL}.csv'
    fpath = os.path.join(DRIVE_PATH, fname)
    if not os.path.exists(fpath):
        print(f'⚠️  {fname} no encontrado — saltando')
        continue

    print(f'\n── {symbol} ──')
    df = pd.read_csv(fpath, index_col=0, parse_dates=True)
    print(f'   {len(df)} barras | {df.index[0].date()} → {df.index[-1].date()}')

    trades, regime_log = run_expanding_window(
        df, symbol, fng_history, p,
        warmup_days=180, retrain_days=90, n_components=4
    )
    metrics = calc_metrics(trades, symbol)
    all_results[symbol] = {'metrics': metrics, 'trades': trades, 'regime_log': regime_log}

    print(f'   Trades: {metrics["trades"]} | Win rate: {metrics["win_rate"]}% | Sharpe: {metrics["sharpe"]} | Retorno: {metrics["return_pct"]:+.1f}%')
    if metrics.get('exit_reasons'):
        print(f'   Salidas: {metrics["exit_reasons"]}')
    if metrics.get('regime_counts'):
        print(f'   Entradas por régimen: {metrics["regime_counts"]}')

print('\n✅ Backtest completado')

In [ ]:
# ── 10. Visualización: Equity curves ─────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Equity Curves — Expanding Window (sin look-ahead)', fontsize=14, color='#e2e8f0')
colors = {'BTCUSDT': '#f59e0b', 'ETHUSDT': '#8b5cf6', 'SOLUSDT': '#10b981', 'XRPUSDT': '#3b82f6'}

for ax, symbol in zip(axes.flat, SYMBOLS):
    if symbol not in all_results:
        ax.set_visible(False); continue
    m   = all_results[symbol]['metrics']
    eq  = m.get('equity', [1000])
    col = colors.get(symbol, '#94a3b8')
    ax.plot(eq, color=col, linewidth=1.5)
    ax.axhline(1000, color='#64748b', linestyle='--', linewidth=0.8)
    ax.fill_between(range(len(eq)), 1000, eq,
                    where=[e >= 1000 for e in eq], alpha=0.15, color='#10b981')
    ax.fill_between(range(len(eq)), 1000, eq,
                    where=[e < 1000 for e in eq], alpha=0.15, color='#ef4444')
    ret_color = '#10b981' if m.get('return_pct', 0) >= 0 else '#ef4444'
    ax.set_title(f"{symbol}  |  Sharpe {m.get('sharpe','—')}  |  "
                 f"Retorno {m.get('return_pct',0):+.1f}%", color=col, fontsize=10)
    ax.set_xlabel('Trades', fontsize=8)
    ax.set_ylabel('Capital ($)', fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/equity_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 11. Visualización: Regímenes en el tiempo (BTC) ───────────
symbol = 'BTCUSDT'
if symbol in all_results:
    reg_log = pd.DataFrame(all_results[symbol]['regime_log']).set_index('ts')
    df_btc  = pd.read_csv(os.path.join(DRIVE_PATH, f'{symbol}_4h.csv'),
                          index_col=0, parse_dates=True)
    df_btc  = df_btc[df_btc.index.isin(reg_log.index)]

    regime_colors = {'BULL_TREND': '#10b981', 'BEAR_TREND': '#ef4444',
                     'SIDEWAYS': '#64748b', 'REVERSAL': '#f59e0b', 'UNKNOWN': '#1e2640'}

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 8), gridspec_kw={'height_ratios': [3, 1]})
    fig.suptitle(f'{symbol} — Precio + Régimen HMM (expanding window)', color='#e2e8f0')

    ax1.plot(df_btc.index, df_btc['close'], color='#f59e0b', linewidth=0.8)

    # Colorear fondo por régimen
    prev_regime, prev_ts = None, None
    for ts, row in reg_log.iterrows():
        if row['regime'] != prev_regime:
            if prev_regime and prev_ts:
                ax1.axvspan(prev_ts, ts, alpha=0.15,
                            color=regime_colors.get(prev_regime, '#1e2640'))
            prev_regime, prev_ts = row['regime'], ts

    ax1.set_ylabel('Precio (USD)', color='#e2e8f0')
    ax1.grid(True, alpha=0.2)

    # Trades encima del precio
    for t in all_results[symbol]['trades']:
        color = '#10b981' if t['result'] == 'WIN' else '#ef4444'
        ax1.scatter(t['entry_time'], t['entry_price'], color=color, s=15, zorder=5)

    # Fear & Greed
    fng_aligned = fng_history.reindex(df_btc.index.normalize(), method='ffill')
    ax2.fill_between(df_btc.index, 0, fng_aligned.values[:len(df_btc)],
                     alpha=0.6, color='#8b5cf6')
    ax2.axhline(20, color='#ef4444', linestyle='--', linewidth=0.7, label='Extreme Fear')
    ax2.axhline(80, color='#10b981', linestyle='--', linewidth=0.7, label='Extreme Greed')
    ax2.set_ylabel('Fear & Greed', color='#e2e8f0')
    ax2.set_ylim(0, 100)
    ax2.legend(fontsize=7)
    ax2.grid(True, alpha=0.2)

    # Leyenda regímenes
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=c, alpha=0.5, label=r)
                       for r, c in regime_colors.items() if r != 'UNKNOWN']
    ax1.legend(handles=legend_elements, loc='upper left', fontsize=7)

    plt.tight_layout()
    plt.savefig('/content/regime_timeline.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ── 12. Resumen final ─────────────────────────────────────────
print('\n' + '='*65)
print('  RESUMEN GLOBAL — EXPANDING WINDOW (sin look-ahead)')
print('='*65)
print(f'{"Par":<12} {"Trades":>8} {"Win%":>8} {"Sharpe":>8} {"Retorno":>10} {"MaxDD":>8}')
print('-'*65)

for symbol in SYMBOLS:
    if symbol not in all_results:
        continue
    m = all_results[symbol]['metrics']
    if m['trades'] == 0:
        print(f'{symbol:<12} {"sin trades":>8}')
        continue
    ret_str = f"{m['return_pct']:+.1f}%"
    print(f"{symbol:<12} {m['trades']:>8} {m['win_rate']:>7.1f}% "
          f"{m['sharpe']:>8.2f} {ret_str:>10} {m['max_drawdown']:>7.1f}%")

print('\n  Motivo de salida (todos los pares):')
all_exits = {}
for r in all_results.values():
    for reason, count in r['metrics'].get('exit_reasons', {}).items():
        all_exits[reason] = all_exits.get(reason, 0) + count
for reason, count in sorted(all_exits.items(), key=lambda x: -x[1]):
    pct = count / sum(all_exits.values()) * 100
    print(f'    {reason:<30} {count:>5} ({pct:.1f}%)')

print('\n  Entradas por régimen (todos los pares):')
all_regimes = {}
for r in all_results.values():
    for reg, count in r['metrics'].get('regime_counts', {}).items():
        all_regimes[reg] = all_regimes.get(reg, 0) + count
for reg, count in sorted(all_regimes.items(), key=lambda x: -x[1]):
    pct = count / sum(all_regimes.values()) * 100
    print(f'    {reg:<20} {count:>5} ({pct:.1f}%)')